# Pre-Plant Crop Yield Prediction Pipeline

This notebook rebuilds the project’s pre-plant modelling workflow from the prepared HarvestStat and annual climate files. It is written for real project data; it does not generate synthetic findings. The app and notebook share the Random Forest model definition in `src.train`; data preparation is described and checked in the notebook and command-line workflow. Run it from the repository root with the project environment active.

**Prediction question:** before planting year *t*, can the crop and state, together with annual rainfall and mean temperature from years *t−1*, *t−2*, and *t−3*, help estimate the recorded yield for year *t*?

HarvestStat Africa v1.2 defines yield in metric tonnes per hectare (mt/ha), with area in hectares and production in metric tonnes. Its QC flags are defined as 0 = no flag, 1 = outlier, and 2 = low variance. QC flags are not predictors; QC-0 results are a sensitivity analysis.

**Previously used evaluation years:** 2021–2023 results were already examined in the earlier notebook. This notebook reproduces that comparison for a traceable workflow; those years must not be described as a fresh, untouched test set or used to choose new model settings.


## 1. Set up the notebook

We import the tools used below and locate the repository from this notebook’s folder. The notebook expects to be saved under `notebooks/` in the project. This cell only locates paths and loads libraries.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "processed" / "climate").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the crop-yield-prediction project folder.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.train import MODEL_FEATURES, make_climate_model as _shared_make_climate_model

CLIMATE_DIR = PROJECT_ROOT / "data" / "processed" / "climate"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"
MODELS_DIR = PROJECT_ROOT / "models"

print("Project folder:", PROJECT_ROOT)
print("scikit-learn version:", sklearn.__version__)


Project folder: /Users/davidoffor/Desktop/crop-yield-prediction
scikit-learn version: 1.9.1


## 2. Load the project inputs

We use the prepared Nigeria HarvestStat file and the verified annual state climate summary. The model target is the recorded yield. We check that the expected fields exist before building any features.

In [2]:
HARVEST_PATH = PROJECT_ROOT / "data" / "processed" / "harveststat_nigeria.csv"
ANNUAL_CLIMATE_PATH = CLIMATE_DIR / "nigeria_state_annual_climate.csv"

harvest = pd.read_csv(HARVEST_PATH)
annual_climate = pd.read_csv(ANNUAL_CLIMATE_PATH)

harvest_required = {"fnid", "admin_1", "product", "season_name", "planting_year", "harvest_year", "yield", "qc_flag"}
climate_required = {"fnid", "admin_1", "year", "annual_rainfall_mm", "T2M_annual_mean"}

missing_harvest = harvest_required - set(harvest.columns)
missing_climate = climate_required - set(annual_climate.columns)
if missing_harvest or missing_climate:
    raise ValueError(f"Missing HarvestStat columns: {sorted(missing_harvest)}; missing climate columns: {sorted(missing_climate)}")

print("HarvestStat:", harvest.shape)
print("Annual climate:", annual_climate.shape)
print("HarvestStat years:", harvest["planting_year"].min(), "to", harvest["planting_year"].max())


HarvestStat: (9636, 16)
Annual climate: (925, 11)
HarvestStat years: 1999 to 2023


## 3. Select the target crops and usable yield records

We keep the eight crops used in the earlier modelling comparison and records with an observed yield. Missing yields cannot be used as supervised-learning examples. We do not remove records based on QC flag here; the flagged records remain in the main analysis and QC-0 is checked separately.

In [3]:
CORE_CROPS = [
    "Cassava",
    "Cowpea",
    "Groundnuts (In Shell)",
    "Maize",
    "Millet",
    "Rice",
    "Sorghum",
    "Yams",
]

crop_targets = harvest.loc[
    harvest["product"].isin(CORE_CROPS)
    & harvest["yield"].notna()
].copy()

crop_targets["planting_year"] = pd.to_numeric(crop_targets["planting_year"], errors="coerce")
crop_targets["yield"] = pd.to_numeric(crop_targets["yield"], errors="coerce")
crop_targets["qc_flag"] = pd.to_numeric(crop_targets["qc_flag"], errors="coerce")
crop_targets = crop_targets.dropna(subset=["planting_year", "yield", "fnid", "product"])
crop_targets["planting_year"] = crop_targets["planting_year"].astype(int)

print("Core-crop yield records:", len(crop_targets))
print("Crops found:", sorted(crop_targets["product"].unique()))
print("States:", crop_targets["fnid"].nunique())
print("Records missing yield after selection:", int(crop_targets["yield"].isna().sum()))


Core-crop yield records: 6004
Crops found: ['Cassava', 'Cowpea', 'Groundnuts (In Shell)', 'Maize', 'Millet', 'Rice', 'Sorghum', 'Yams']
States: 37
Records missing yield after selection: 0


## 4. Check the annual climate keys

A state must have at most one annual climate row per year. We confirm this before calculating lagged features, because duplicate state-year rows could multiply records in the later join.

In [4]:
annual_climate = annual_climate.copy()
annual_climate["year"] = pd.to_numeric(annual_climate["year"], errors="coerce")
annual_climate["annual_rainfall_mm"] = pd.to_numeric(annual_climate["annual_rainfall_mm"], errors="coerce")
annual_climate["T2M_annual_mean"] = pd.to_numeric(annual_climate["T2M_annual_mean"], errors="coerce")
annual_climate = annual_climate.dropna(subset=["fnid", "year", "annual_rainfall_mm", "T2M_annual_mean"]).copy()
annual_climate["year"] = annual_climate["year"].astype(int)

if annual_climate.duplicated(["fnid", "year"]).any():
    raise ValueError("Annual climate has duplicate state-year rows.")

print("Annual climate rows:", len(annual_climate))
print("States:", annual_climate["fnid"].nunique())
print("Climate years:", annual_climate["year"].min(), "to", annual_climate["year"].max())


Annual climate rows: 925
States: 37
Climate years: 1999 to 2023


## 5. Build climate features available before planting

For each state and planting year *t*, the three-year features are averages over *t−1*, *t−2*, and *t−3*. We shift climate values back one year before taking the three-year rolling average. This prevents weather from the planting year or later from entering the predictors.

In [5]:
annual_features = annual_climate.sort_values(["fnid", "year"]).copy()

grouped = annual_features.groupby("fnid", sort=False)
annual_features["prior3y_mean_rainfall"] = grouped["annual_rainfall_mm"].transform(
    lambda values: values.shift(1).rolling(window=3, min_periods=3).mean()
)
annual_features["prior3y_mean_T2M"] = grouped["T2M_annual_mean"].transform(
    lambda values: values.shift(1).rolling(window=3, min_periods=3).mean()
)

features_for_planting = annual_features.rename(columns={"year": "planting_year"})[
    ["fnid", "admin_1", "planting_year", "prior3y_mean_rainfall", "prior3y_mean_T2M"]
].copy()
features_for_planting["climate_data_through_year"] = features_for_planting["planting_year"] - 1

print(features_for_planting.dropna().head().to_string(index=False))


      fnid admin_1  planting_year  prior3y_mean_rainfall  prior3y_mean_T2M  climate_data_through_year
NG1996A101    Abia           2002            2309.631369         25.126252                       2001
NG1996A101    Abia           2003            2220.851908         25.063640                       2002
NG1996A101    Abia           2004            2282.344354         25.242237                       2003
NG1996A101    Abia           2005            2293.627035         25.410877                       2004
NG1996A101    Abia           2006            2323.913704         25.560356                       2005


## 6. Join yields to pre-plant climate and verify timing

We join each crop-state planting-year yield record to that state’s prior-three-year climate summary. The join is many-to-one: several crops can share the same state-year climate features, but the climate table must have only one row per state-year.

In [6]:
model_data = crop_targets.merge(
    features_for_planting,
    on=["fnid", "admin_1", "planting_year"],
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("Join results:")
print(model_data["_merge"].value_counts().to_string())
model_data = model_data.loc[model_data["_merge"] == "both"].drop(columns="_merge").copy()
model_data = model_data.dropna(subset=["prior3y_mean_rainfall", "prior3y_mean_T2M"]).copy()

assert (model_data["climate_data_through_year"] < model_data["planting_year"]).all()
assert model_data[["prior3y_mean_rainfall", "prior3y_mean_T2M"]].notna().all().all()
assert not model_data.duplicated(["fnid", "product", "planting_year", "season_name"]).any()

print("Model rows with three prior climate years:", len(model_data))
print("Planting years:", model_data["planting_year"].min(), "to", model_data["planting_year"].max())
print("Rows by crop:")
print(model_data.groupby("product").size().to_string())


Join results:
_merge
both          6004
left_only        0
right_only       0
Model rows with three prior climate years: 5284
Planting years: 2002 to 2023
Rows by crop:
product
Cassava                  749
Cowpea                   740
Groundnuts (In Shell)    667
Maize                    808
Millet                   439
Rice                     798
Sorghum                  473
Yams                     610


## 7. Confirm the time periods and QC composition

We keep the previously agreed chronological periods: 2000–2018 for model development, 2019–2020 for validation, and 2021–2023 for the earlier test-period comparison. The test-period results are already known from prior exploration, so this notebook labels them as a reproduction rather than a fresh test.

In [7]:
def assign_period(year):
    if 2000 <= year <= 2018:
        return "train 2000-2018"
    if 2019 <= year <= 2020:
        return "validation 2019-2020"
    if 2021 <= year <= 2023:
        return "previously examined period 2021-2023"
    return "outside planned periods"

model_data["period"] = model_data["planting_year"].map(assign_period)
print("Rows by period:")
print(model_data["period"].value_counts().to_string())
print("QC flags by period:")
print(model_data.groupby(["period", "qc_flag"], dropna=False).size().unstack(fill_value=0).to_string())
print("Rows by crop and period:")
print(pd.crosstab(model_data["product"], model_data["period"]).to_string())


Rows by period:
period
train 2000-2018                         4075
previously examined period 2021-2023     726
validation 2019-2020                     483
QC flags by period:
qc_flag                                  0   1    2
period                                             
previously examined period 2021-2023   679   0   47
train 2000-2018                       3816  16  243
validation 2019-2020                   381   0  102
Rows by crop and period:
period                 previously examined period 2021-2023  train 2000-2018  validation 2019-2020
product                                                                                           
Cassava                                                 102              579                    68
Cowpea                                                  111              555                    74
Groundnuts (In Shell)                                    87              523                    57
Maize                                     

## 8. Define predictors and check for leakage

The predictors are crop, state, and the two prior-three-year climate averages. Planting year is used only to define the time periods. Yield is the target. Production, area, and QC flag are excluded from predictors; the QC flag is used only to define a sensitivity subset.

In [8]:
TARGET = "yield"
CATEGORICAL_FEATURES = ["product", "fnid"]
NUMERIC_FEATURES = ["prior3y_mean_rainfall", "prior3y_mean_T2M"]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

for forbidden in ["production", "area", "qc_flag", "planting_year", "harvest_year", TARGET]:
    assert forbidden not in FEATURES, f"Forbidden or non-predictor field included: {forbidden}"

assert np.isfinite(model_data[NUMERIC_FEATURES].to_numpy(dtype=float)).all()
print("Predictors:", FEATURES)
print("Target:", TARGET)
print("Area/production used as predictors:", bool({"area", "production"} & set(FEATURES)))
print("QC flag used as predictor:", "qc_flag" in FEATURES)


Predictors: ['product', 'fnid', 'prior3y_mean_rainfall', 'prior3y_mean_T2M']
Target: yield
Area/production used as predictors: False
QC flag used as predictor: False


## 9. Define the two comparison methods and scoring

The baseline predicts the median yield for that crop using training records only. A global training median is the fallback if a crop has no training records. The climate model is the previously examined Random Forest specification. Both methods are scored on the same records using MAE, RMSE, and R². HarvestStat Africa v1.2 defines yield in metric tonnes per hectare (mt/ha), so MAE and RMSE are reported in mt/ha.

In [9]:
RANDOM_SEED = 42
RF_PARAMETERS = {"n_estimators": 300, "min_samples_leaf": 5, "random_state": RANDOM_SEED, "n_jobs": -1}


def regression_scores(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return {
        "records": int(len(actual)),
        "MAE": float(mean_absolute_error(actual, predicted)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "R2": float(r2_score(actual, predicted)) if len(actual) > 1 and np.var(actual) > 0 else np.nan,
    }


def make_climate_model():
    return _shared_make_climate_model()


def fit_and_score(train_frame, evaluation_frame, evaluation_label):
    if train_frame.empty or evaluation_frame.empty:
        raise ValueError(f"Empty training or evaluation data for {evaluation_label}.")

    crop_medians = train_frame.groupby("product")[TARGET].median()
    overall_median = float(train_frame[TARGET].median())
    baseline_prediction = evaluation_frame["product"].map(crop_medians).fillna(overall_median).to_numpy()

    climate_model = make_climate_model()
    climate_model.fit(train_frame[FEATURES], train_frame[TARGET])
    climate_prediction = climate_model.predict(evaluation_frame[FEATURES])

    prediction_frame = evaluation_frame[["fnid", "admin_1", "product", "planting_year", TARGET, "qc_flag"]].copy()
    prediction_frame = prediction_frame.rename(columns={TARGET: "actual_yield"})
    prediction_frame["crop_median_prediction"] = baseline_prediction
    prediction_frame["climate_model_prediction"] = climate_prediction
    prediction_frame["evaluation_period"] = evaluation_label

    comparisons = []
    groups = [("Overall", prediction_frame)]
    groups.extend((crop, sub) for crop, sub in prediction_frame.groupby("product", sort=True))
    for group_name, sub in groups:
        for model_name, prediction_column in [
            ("Crop median baseline", "crop_median_prediction"),
            ("Crop, state, and prior-three-year climate", "climate_model_prediction"),
        ]:
            comparisons.append({
                "evaluation_period": evaluation_label,
                "group": group_name,
                "model": model_name,
                **regression_scores(sub["actual_yield"], sub[prediction_column]),
            })

    return pd.DataFrame(comparisons), prediction_frame, climate_model, crop_medians, overall_median


## 10. Compare models on the validation years

The validation period is 2019–2020. Models are trained on 2000–2018 only. We use this comparison to document how the two predeclared methods performed during development. The 2021–2023 period is not used here.

In [10]:
train_rows = model_data.loc[model_data["period"] == "train 2000-2018"].copy()
validation_rows = model_data.loc[model_data["period"] == "validation 2019-2020"].copy()

validation_scores, validation_predictions, validation_climate_model, _, _ = fit_and_score(
    train_rows, validation_rows, "validation 2019-2020"
)

print("Training rows:", len(train_rows))
print("Validation rows:", len(validation_rows))
display(validation_scores.round(3))


Training rows: 4075
Validation rows: 483


,evaluation_period,group,model,records,MAE,RMSE,R2
0,validation 2019-2020,Overall,Crop median baseline,483,1.164,2.278,0.513
1,validation 2019-2020,Overall,"Crop, state, and prior-three-year climate",483,1.205,2.726,0.302
2,validation 2019-2020,Cassava,Crop median baseline,68,4.014,4.760,-1.143
3,validation 2019-2020,Cassava,"Crop, state, and prior-three-year climate",68,4.171,5.776,-2.156
4,validation 2019-2020,Cowpea,Crop median baseline,74,0.262,0.371,-0.294
5,validation 2019-2020,Cowpea,"Crop, state, and prior-three-year climate",74,0.247,0.369,-0.282
6,validation 2019-2020,Groundnuts (In Shell),Crop median baseline,57,0.300,0.429,-0.130
7,validation 2019-2020,Groundnuts (In Shell),"Crop, state, and prior-three-year climate",57,0.288,0.375,0.139
8,validation 2019-2020,Maize,Crop median baseline,74,0.441,0.546,-0.902
9,validation 2019-2020,Maize,"Crop, state, and prior-three-year climate",74,0.383,0.502,-0.609


## 11. Run the QC-0 validation sensitivity check

We repeat the same validation comparison after restricting both training and validation records to QC flag 0. The QC flag still is not a predictor. This comparison answers whether the validation result changes when the analysis is limited to records carrying flag 0; it does not establish the meaning of that flag.

In [11]:
train_qc0 = train_rows.loc[train_rows["qc_flag"] == 0].copy()
validation_qc0 = validation_rows.loc[validation_rows["qc_flag"] == 0].copy()

validation_qc0_scores, validation_qc0_predictions, validation_qc0_model, _, _ = fit_and_score(
    train_qc0, validation_qc0, "validation 2019-2020, QC-0 only"
)

print("QC-0 training rows:", len(train_qc0))
print("QC-0 validation rows:", len(validation_qc0))
display(validation_qc0_scores.round(3))


QC-0 training rows: 3816
QC-0 validation rows: 381


,evaluation_period,group,model,records,MAE,RMSE,R2
0,"validation 2019-2020, QC-0 only",Overall,Crop median baseline,381,0.892,1.877,0.677
1,"validation 2019-2020, QC-0 only",Overall,"Crop, state, and prior-three-year climate",381,0.873,1.974,0.643
2,"validation 2019-2020, QC-0 only",Cassava,Crop median baseline,38,3.257,4.230,-0.890
3,"validation 2019-2020, QC-0 only",Cassava,"Crop, state, and prior-three-year climate",38,3.402,4.302,-0.955
4,"validation 2019-2020, QC-0 only",Cowpea,Crop median baseline,64,0.283,0.398,-0.393
5,"validation 2019-2020, QC-0 only",Cowpea,"Crop, state, and prior-three-year climate",64,0.244,0.379,-0.262
6,"validation 2019-2020, QC-0 only",Groundnuts (In Shell),Crop median baseline,45,0.308,0.448,-0.291
7,"validation 2019-2020, QC-0 only",Groundnuts (In Shell),"Crop, state, and prior-three-year climate",45,0.313,0.402,-0.040
8,"validation 2019-2020, QC-0 only",Maize,Crop median baseline,54,0.423,0.531,-1.034
9,"validation 2019-2020, QC-0 only",Maize,"Crop, state, and prior-three-year climate",54,0.362,0.478,-0.651


## 12. Reproduce the previously examined 2021–2023 comparison

For comparability with the earlier experiment, we train on planting years 2000–2020 and evaluate on 2021–2023. The model settings are fixed above; do not change them based on these results. Because this period was already examined, treat these scores as a reproducibility check and previously examined period—not as an untouched final test.

In [12]:
historical_train = model_data.loc[model_data["planting_year"].between(2000, 2020)].copy()
historical_period = model_data.loc[model_data["planting_year"].between(2021, 2023)].copy()

historical_scores, historical_predictions, historical_climate_model, final_crop_medians, final_overall_median = fit_and_score(
    historical_train, historical_period, "previously examined period 2021-2023"
)

print("Training rows, 2000-2020:", len(historical_train))
print("Previously examined evaluation rows, 2021-2023:", len(historical_period))
display(historical_scores.round(3))


Training rows, 2000-2020: 4558
Previously examined evaluation rows, 2021-2023: 726


,evaluation_period,group,model,records,MAE,RMSE,R2
0,previously examined period 2021-2023,Overall,Crop median baseline,726,1.106,2.175,0.609
1,previously examined period 2021-2023,Overall,"Crop, state, and prior-three-year climate",726,1.146,2.623,0.431
2,previously examined period 2021-2023,Cassava,Crop median baseline,102,3.546,4.307,-0.567
3,previously examined period 2021-2023,Cassava,"Crop, state, and prior-three-year climate",102,3.892,5.535,-1.587
4,previously examined period 2021-2023,Cowpea,Crop median baseline,111,0.252,0.358,-0.151
5,previously examined period 2021-2023,Cowpea,"Crop, state, and prior-three-year climate",111,0.254,0.347,-0.080
6,previously examined period 2021-2023,Groundnuts (In Shell),Crop median baseline,87,0.277,0.390,-0.007
7,previously examined period 2021-2023,Groundnuts (In Shell),"Crop, state, and prior-three-year climate",87,0.273,0.352,0.182
8,previously examined period 2021-2023,Maize,Crop median baseline,111,0.476,0.590,-0.731
9,previously examined period 2021-2023,Maize,"Crop, state, and prior-three-year climate",111,0.438,0.566,-0.591


## 13. Repeat the 2021–2023 comparison on QC-0 records

This is the QC-0 sensitivity result for the same previously examined period. It remains a descriptive reproduction of an already reviewed comparison and must not be used to tune or select settings.

In [13]:
historical_train_qc0 = historical_train.loc[historical_train["qc_flag"] == 0].copy()
historical_period_qc0 = historical_period.loc[historical_period["qc_flag"] == 0].copy()

historical_qc0_scores, historical_qc0_predictions, historical_qc0_model, _, _ = fit_and_score(
    historical_train_qc0, historical_period_qc0, "previously examined period 2021-2023, QC-0 only"
)

print("QC-0 training rows:", len(historical_train_qc0))
print("QC-0 evaluation rows:", len(historical_period_qc0))
display(historical_qc0_scores.round(3))


QC-0 training rows: 4197
QC-0 evaluation rows: 679


,evaluation_period,group,model,records,MAE,RMSE,R2
0,"previously examined period 2021-2023, QC-0 only",Overall,Crop median baseline,679,1.023,2.060,0.651
1,"previously examined period 2021-2023, QC-0 only",Overall,"Crop, state, and prior-three-year climate",679,1.040,2.361,0.541
2,"previously examined period 2021-2023, QC-0 only",Cassava,Crop median baseline,90,3.485,4.304,-0.627
3,"previously examined period 2021-2023, QC-0 only",Cassava,"Crop, state, and prior-three-year climate",90,3.734,4.910,-1.118
4,"previously examined period 2021-2023, QC-0 only",Cowpea,Crop median baseline,106,0.257,0.365,-0.166
5,"previously examined period 2021-2023, QC-0 only",Cowpea,"Crop, state, and prior-three-year climate",106,0.239,0.328,0.060
6,"previously examined period 2021-2023, QC-0 only",Groundnuts (In Shell),Crop median baseline,82,0.274,0.390,-0.006
7,"previously examined period 2021-2023, QC-0 only",Groundnuts (In Shell),"Crop, state, and prior-three-year climate",82,0.290,0.367,0.110
8,"previously examined period 2021-2023, QC-0 only",Maize,Crop median baseline,104,0.476,0.593,-0.769
9,"previously examined period 2021-2023, QC-0 only",Maize,"Crop, state, and prior-three-year climate",104,0.455,0.582,-0.706


## 14. Inspect prediction errors by crop

The overall score combines crops with very different yield scales. We display crop-level comparisons alongside the overall results so the pooled metric is not mistaken for equal performance across crops. These results describe model errors; they do not establish causes.

In [14]:
overall_and_crop_results = pd.concat(
    [validation_scores, validation_qc0_scores, historical_scores, historical_qc0_scores],
    ignore_index=True,
)
display(overall_and_crop_results.loc[overall_and_crop_results["group"] == "Overall"].round(3))
print("Previously examined 2021-2023 results by crop:")
display(
    historical_scores.loc[historical_scores["group"] != "Overall"]
    .sort_values(["group", "model"])
    .round(3)
)


,evaluation_period,group,model,records,MAE,RMSE,R2
0,validation 2019-2020,Overall,Crop median baseline,483,1.164,2.278,0.513
1,validation 2019-2020,Overall,"Crop, state, and prior-three-year climate",483,1.205,2.726,0.302
18,"validation 2019-2020, QC-0 only",Overall,Crop median baseline,381,0.892,1.877,0.677
19,"validation 2019-2020, QC-0 only",Overall,"Crop, state, and prior-three-year climate",381,0.873,1.974,0.643
36,previously examined period 2021-2023,Overall,Crop median baseline,726,1.106,2.175,0.609
37,previously examined period 2021-2023,Overall,"Crop, state, and prior-three-year climate",726,1.146,2.623,0.431
54,"previously examined period 2021-2023, QC-0 only",Overall,Crop median baseline,679,1.023,2.060,0.651
55,"previously examined period 2021-2023, QC-0 only",Overall,"Crop, state, and prior-three-year climate",679,1.040,2.361,0.541


Previously examined 2021-2023 results by crop:


,evaluation_period,group,model,records,MAE,RMSE,R2
2,previously examined period 2021-2023,Cassava,Crop median baseline,102,3.546,4.307,-0.567
3,previously examined period 2021-2023,Cassava,"Crop, state, and prior-three-year climate",102,3.892,5.535,-1.587
4,previously examined period 2021-2023,Cowpea,Crop median baseline,111,0.252,0.358,-0.151
5,previously examined period 2021-2023,Cowpea,"Crop, state, and prior-three-year climate",111,0.254,0.347,-0.080
6,previously examined period 2021-2023,Groundnuts (In Shell),Crop median baseline,87,0.277,0.390,-0.007
7,previously examined period 2021-2023,Groundnuts (In Shell),"Crop, state, and prior-three-year climate",87,0.273,0.352,0.182
8,previously examined period 2021-2023,Maize,Crop median baseline,111,0.476,0.590,-0.731
9,previously examined period 2021-2023,Maize,"Crop, state, and prior-three-year climate",111,0.438,0.566,-0.591
10,previously examined period 2021-2023,Millet,Crop median baseline,60,0.289,0.430,-0.067
11,previously examined period 2021-2023,Millet,"Crop, state, and prior-three-year climate",60,0.343,0.482,-0.340


## 15. Refit the crop-median baseline on all available eligible records

The earlier comparisons estimate how the fixed methods performed under the stated time periods. For a reusable baseline prediction rule, we now calculate crop medians from all eligible records available in this dataset. This refit is for future use and is not an evaluation score. The crop median remains the reference model because the earlier overall comparisons did not show a consistent advantage for the climate model.

In [15]:
deployment_crop_medians = model_data.groupby("product")[TARGET].median().sort_index()
deployment_overall_median = float(model_data[TARGET].median())

print("Crop medians learned from all eligible records:")
display(deployment_crop_medians.rename("yield_median").to_frame())
print("Fallback overall median:", round(deployment_overall_median, 4))


Crop medians learned from all eligible records:


,yield_median
product,
Cassava,9.419626
Cowpea,0.750000
Groundnuts (In Shell),1.165269
Maize,1.776430
Millet,0.950226
Rice,1.981494
Sorghum,1.189516
Yams,9.668797


Fallback overall median: 1.5998


## 16. Save reproducible tables and the baseline prediction rule

We save validation and previously examined-period score tables, prediction-level results, and the crop-median baseline values. The model metadata records the feature definition, split periods, model settings, and the documented yield unit and QC flag meanings. These files are real-data outputs only.

In [16]:
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

all_scores = pd.concat(
    [validation_scores, validation_qc0_scores, historical_scores, historical_qc0_scores],
    ignore_index=True,
)
all_predictions = pd.concat(
    [validation_predictions, validation_qc0_predictions, historical_predictions, historical_qc0_predictions],
    ignore_index=True,
)

all_scores.to_csv(TABLES_DIR / "preplant_model_comparison.csv", index=False)
all_predictions.to_csv(TABLES_DIR / "preplant_model_predictions.csv", index=False)

dep_baseline = deployment_crop_medians.rename("crop_median_yield").reset_index()
dep_baseline.to_csv(TABLES_DIR / "preplant_crop_median_baseline.csv", index=False)

metadata = {
    "target": "HarvestStat yield in metric tonnes per hectare (mt/ha), per HarvestStat Africa v1.2 documentation",
    "prediction_timing": "before planting year; climate predictors use only years t-1, t-2, and t-3",
    "climate_features": {
        "prior3y_mean_rainfall": "mean of annual state rainfall totals for the three complete years before planting",
        "prior3y_mean_T2M": "mean of annual state mean temperature for the three complete years before planting",
    },
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "excluded_predictors": ["production", "area", "qc_flag", "planting_year", "harvest_year"],
    "validation_years": [2019, 2020],
    "previously_examined_evaluation_years": [2021, 2022, 2023],
    "previously_examined_period_is_untouched_test": False,
    "random_forest_parameters": RF_PARAMETERS,
    "qc_flag_definition": "0 = no flag; 1 = outlier; 2 = low variance; QC-0 results are a sensitivity analysis only",
    "yield_unit": "metric tonnes per hectare (mt/ha), per HarvestStat Africa v1.2 documentation",
    "python_version": sys.version.split()[0],
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "baseline_fallback_overall_median": deployment_overall_median,
}
(MODELS_DIR / "preplant_pipeline_metadata.json").write_text(json.dumps(metadata, indent=2))

print("Saved:")
print(TABLES_DIR / "preplant_model_comparison.csv")
print(TABLES_DIR / "preplant_model_predictions.csv")
print(TABLES_DIR / "preplant_crop_median_baseline.csv")
print(MODELS_DIR / "preplant_pipeline_metadata.json")


Saved:
/Users/davidoffor/Desktop/crop-yield-prediction/results/tables/preplant_model_comparison.csv
/Users/davidoffor/Desktop/crop-yield-prediction/results/tables/preplant_model_predictions.csv
/Users/davidoffor/Desktop/crop-yield-prediction/results/tables/preplant_crop_median_baseline.csv
/Users/davidoffor/Desktop/crop-yield-prediction/models/preplant_pipeline_metadata.json


## 17. Findings and limitations


This notebook compared a crop-median baseline with a Random Forest model using crop, state, and average annual rainfall and mean temperature from the three years before planting. Production, area, and QC flag were not used as predictors.

| Evaluation period | Records | Model | MAE | RMSE | R² |
|---|---:|---|---:|---:|---:|
| Validation, all records | 483 | Crop-median baseline | 1.164 | 2.278 | 0.513 |
| Validation, all records | 483 | Crop, state, and climate | 1.205 | 2.726 | 0.302 |
| Validation, QC-0 only | 381 | Crop-median baseline | 0.892 | 1.877 | 0.677 |
| Validation, QC-0 only | 381 | Crop, state, and climate | 0.873 | 1.974 | 0.643 |
| Previously examined 2021–2023, all records | 726 | Crop-median baseline | 1.106 | 2.175 | 0.609 |
| Previously examined 2021–2023, all records | 726 | Crop, state, and climate | 1.146 | 2.623 | 0.431 |
| Previously examined 2021–2023, QC-0 only | 679 | Crop-median baseline | 1.023 | 2.060 | 0.651 |
| Previously examined 2021–2023, QC-0 only | 679 | Crop, state, and climate | 1.040 | 2.361 | 0.541 |

On the validation records overall, the crop-median baseline performed better than the climate model on all three measures. In the QC-0 validation subset, the climate model had slightly lower MAE, while the baseline had lower RMSE and higher R². In the previously examined 2021–2023 period, the crop-median baseline performed better overall on all three measures, both with all eligible records and with QC-0 records only.

Results differed by crop, but the climate model did not show a consistent overall improvement over the baseline. These results apply to the specific data, features, and model tested here; they do not show that climate is unrelated to yield.

## Limitations

- HarvestStat Africa v1.2 documents yield in metric tonnes per hectare (mt/ha), so MAE and RMSE are reported in mt/ha.
- HarvestStat defines QC flags as 0 = no flag, 1 = outlier, and 2 = low variance. QC-0 results are a sensitivity analysis; “no flag” does not guarantee error-free records.
- The 2021–2023 period was examined in an earlier experiment. The reproduced scores are not from an untouched final test set and should not be used to tune model settings.
- The climate features summarize state-level annual rainfall and temperature. They do not capture within-state differences or specific conditions during each crop’s growing season.
- The model does not include other influences on yield, such as soil, seed varieties, farm inputs, pests, or management.
- The model results describe predictive performance and associations; they do not show that climate caused a yield change.
